# MLP Baseline Notebook

单模型快速验证入口（复用共享函数）。

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("nbformat") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])

%run ./functions.ipynb

def _find_root() -> Path:
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "notebooks").exists() and (p / "requirements.txt").exists():
            return p
    return c

REPO_ROOT = _find_root()
cache = torch.load(REPO_ROOT / "log" / "notebook_cache" / "prepared_data.pt")
prepared = PreparedData(
    dense=cache["dense"],
    sparse=cache["sparse"],
    labels=cache["labels"],
    sample_ids=cache["sample_ids"],
    sparse_fields=cache["sparse_fields"],
    dense_fields=cache["dense_fields"],
)

train_loader, train_eval_loader, val_loader = make_loaders(prepared, batch_size=256, val_ratio=0.2, seed=42)
model = build_model("MLP", prepared.sparse_fields, prepared.dense.shape[1], int(cache["num_buckets"]))
hist, _ = train_model(model, train_loader, val_loader, epochs=3, lr=1e-3, device=("cuda" if torch.cuda.is_available() else "cpu"))
hist